In [5]:
TEST_ENTITIES = {
    "LOC": ["Gaillimh", "Corcaigh", "Éire", "Doire", "Baile Átha Cliath"],
    "PER": ["Seán Ó Briain", "Tomás Mac Cárthaigh", "Máire Ní Bhriain", "Pádraig Ó Néill"],
    "ORG": ["Sinn Féin", "Fianna Fáil", "RTÉ"],
}

In [6]:
# CELL 1 — Imports and paths

import pandas as pd
from collections import defaultdict

BASE = "/kaggle/input/datasets/michaelmarkey64"
LOC_LOGAINM_PATH = f"{BASE}/irish-ner-kg-consolidated/loc_logainm_enriched.csv"

# ── Test entities ─────────────────────────────────────────────────────────────

TEST_ENTITIES = {
    "LOC": ["Gaillimh", "Corcaigh", "Éire", "Doire", "Baile Átha Cliath"],
    "PER": ["Seán Ó Briain", "Tomás Mac Cárthaigh", "Máire Ní Bhriain", "Pádraig Ó Néill"],
    "ORG": ["Sinn Féin", "Fianna Fáil", "RTÉ"],
}

In [8]:
# CELL 2 — Layer 1: Logainm lookup (LOC only)

def build_logainm_lookup(path):
    df = pd.read_csv(path)
    print(f"Loaded {len(df)} LOC entries from Logainm enriched CSV")
    print(f"Columns: {df.columns.tolist()}")

    lookup = defaultdict(set)
    for _, row in df.iterrows():
        key = str(row.get("name_ga_lg", "") or "").strip()
        if not key or key.lower() == "nan":
            continue
        for col in ["name_ga_lg", "genitive_lg"]:
            val = str(row.get(col, "") or "").strip()
            if val and val.lower() != "nan":
                lookup[key].add(val)

    return lookup

logainm_lookup = build_logainm_lookup(LOC_LOGAINM_PATH)
print(f"\nLogainm lookup: {len(logainm_lookup)} LOC entries")

# ── Layer 1 function ──────────────────────────────────────────────────────────

def layer1_logainm(entity, label):
    """Return set of attested Logainm forms for a LOC entity, empty set otherwise."""
    if label != "LOC":
        return set()
    return logainm_lookup.get(entity, set())

# ── Test ──────────────────────────────────────────────────────────────────────

print("\n── Layer 1 (Logainm) test ──")
for entity in TEST_ENTITIES["LOC"]:
    forms = layer1_logainm(entity, "LOC")
    status = f"{len(forms)} forms: {sorted(forms)}" if forms else "MISS"
    print(f"  {entity:25s} → {status}")

print()
for entity in TEST_ENTITIES["PER"] + TEST_ENTITIES["ORG"]:
    forms = layer1_logainm(entity, "PER")
    print(f"  {entity:25s} → (skipped — not LOC)")

Loaded 185 LOC entries from Logainm enriched CSV
Columns: ['qid', 'logainm_id_new', 'name_ga_lg', 'name_en_lg', 'genitive_lg', 'county_ga', 'gaeltacht', 'category_lg']

Logainm lookup: 117 LOC entries

── Layer 1 (Logainm) test ──
  Gaillimh                  → 2 forms: ['Gaillimh', 'na Gaillimhe']
  Corcaigh                  → 2 forms: ['Chorcaí', 'Corcaigh']
  Éire                      → MISS
  Doire                     → 2 forms: ['Dhoire', 'Doire']
  Baile Átha Cliath         → 2 forms: ['Baile Átha Cliath', 'Bhaile Átha Cliath']

  Seán Ó Briain             → (skipped — not LOC)
  Tomás Mac Cárthaigh       → (skipped — not LOC)
  Máire Ní Bhriain          → (skipped — not LOC)
  Pádraig Ó Néill           → (skipped — not LOC)
  Sinn Féin                 → (skipped — not LOC)
  Fianna Fáil               → (skipped — not LOC)
  RTÉ                       → (skipped — not LOC)


In [25]:
# CELL 3 — Layer 2: UD Irish-IDT attested variant lookup
import requests, os
from collections import defaultdict
 
UD_DIR = "/tmp/ud_idt"
os.makedirs(UD_DIR, exist_ok=True)
 
UD_URLS = {
    "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_Irish-IDT/master/ga_idt-ud-train.conllu",
    "dev":   "https://raw.githubusercontent.com/UniversalDependencies/UD_Irish-IDT/master/ga_idt-ud-dev.conllu",
    "test":  "https://raw.githubusercontent.com/UniversalDependencies/UD_Irish-IDT/master/ga_idt-ud-test.conllu",
}
 
# ── Download ──────────────────────────────────────────────────────────────────
for split, url in UD_URLS.items():
    path = f"{UD_DIR}/ga_idt-ud-{split}.conllu"
    if not os.path.exists(path):
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            with open(path, "w", encoding="utf-8") as f:
                f.write(r.text)
            print(f"Downloaded {split} ({len(r.text):,} chars)")
        else:
            print(f"FAILED {split}: HTTP {r.status_code}")
    else:
        print(f"Already exists: {split}")
 
# ── Build lemma → surface variants lookup ─────────────────────────────────────
def build_ud_lookup(ud_dir):
    """
    Parse all UD Irish-IDT splits and build a dict of
    lemma -> set of attested surface forms for PROPN tokens.
    Filters out all-caps tokens (treebank heading artefacts).
    """
    lookup = defaultdict(set)
    for fname in os.listdir(ud_dir):
        if not fname.endswith(".conllu"):
            continue
        with open(os.path.join(ud_dir, fname), encoding="utf-8") as f:
            for line in f:
                line = line.rstrip()
                if not line or line.startswith("#"):
                    continue
                parts = line.split("\t")
                if len(parts) < 10 or not parts[0].isdigit():
                    continue
                token, lemma, upos = parts[1], parts[2], parts[3]
                if upos != "PROPN" or not lemma or not token:
                    continue
                # skip all-caps tokens — treebank heading artefacts
                if token == token.upper() and len(token) > 2:
                    continue
                # skip tokens where only case differs from lemma (not a real variant)
                if token.lower() == lemma.lower() and token != lemma:
                    continue
                if lemma != token:
                    lookup[lemma].add(token)
                lookup[lemma].add(lemma)  # always include nominative
    return lookup
 
ud_lookup = build_ud_lookup(UD_DIR)
print(f"\nUD lookup: {len(ud_lookup)} PROPN lemmas with attested variants")
multi = {k: v for k, v in ud_lookup.items() if len(v) > 1}
print(f"Lemmas with >1 attested form: {len(multi)}")
 
# ── Layer 2 function ──────────────────────────────────────────────────────────
def layer2_ud(entity, label):
    """Return set of attested UD surface forms for an entity, all types."""
    return ud_lookup.get(entity, set())
 
# ── Test ──────────────────────────────────────────────────────────────────────
print("\n── Layer 2 (UD Irish-IDT) test ──")
for label, entities in TEST_ENTITIES.items():
    print(f"\n  {label}:")
    for entity in entities:
        forms = layer2_ud(entity, label)
        status = f"{len(forms)} forms: {sorted(forms)}" if forms else "MISS"
        print(f"    {entity:30s} → {status}")
 

Already exists: train
Already exists: dev
Already exists: test

UD lookup: 1792 PROPN lemmas with attested variants
Lemmas with >1 attested form: 556

── Layer 2 (UD Irish-IDT) test ──

  LOC:
    Gaillimh                       → 4 forms: ['Gaillimh', 'Gaillimhe', 'Ghaillimh', 'nGaillimh']
    Corcaigh                       → 4 forms: ['Chorcaigh', 'Chorcaí', 'Corcaigh', 'gCorcaigh']
    Éire                           → 7 forms: ['hÉire', 'hÉireann', 'hÉirinn', 'Éire', 'Éireann', 'Éireanna', 'Éirinn']
    Doire                          → 4 forms: ['Dhoire', 'Dhoirí', 'Doire', 'nDoire']
    Baile Átha Cliath              → MISS

  PER:
    Seán Ó Briain                  → MISS
    Tomás Mac Cárthaigh            → MISS
    Máire Ní Bhriain               → MISS
    Pádraig Ó Néill                → MISS

  ORG:
    Sinn Féin                      → MISS
    Fianna Fáil                    → MISS
    RTÉ                            → MISS


In [10]:
# CELL 4 — Layer 3: Manual genitive lexicon

# Covers irregular genitives that rule-based expansion cannot generate.
# Keyed on the nominative form (or first token of a multi-token name).
# Values are sets of attested alternative forms.

MANUAL_GENITIVE_LEXICON = {
    # ── First names — masculine ───────────────────────────────────────────────
    "Seán":       {"Sheáin"},
    "Tomás":      {"Thomáis"},
    "Séamas":     {"Shéamais"},
    "Micheál":    {"Mhichíl"},
    "Cathal":     {"Cathail"},
    "Niall":      {"Néill"},
    "Aodh":       {"Aodha"},
    "Ciarán":     {"Chiaráin"},
    "Ruairí":     {"Ruairí"},
    "Pádraig":    {"Phádraig"},
    "Breandán":   {"Bhreandáin"},
    "Colm":       {"Cholm"},
    "Conn":       {"Cuinn"},
    "Diarmuid":   {"Diarmada"},
    "Donnchadh":  {"Donnchaidh"},
    "Eoin":       {"Eoin"},
    "Fearghus":   {"Fearghusa"},
    "Fionn":      {"Finn"},
    "Labhras":    {"Labhráis"},
    "Muiris":     {"Mhuiris"},
    "Peadar":     {"Pheadair"},
    "Proinsias":  {"Proinsiis"},
    "Rónán":      {"Rónáin"},
    "Seamus":     {"Séamuis"},
    "Tadhg":      {"Taidhg"},
    "Uilliam":    {"Uilliam"},

    # ── First names — feminine ────────────────────────────────────────────────
    "Máire":      {"Mháire"},
    "Bríd":       {"Bhríde"},
    "Siobhán":    {"Shiobháin"},
    "Áine":       {"Áine"},
    "Aoife":      {"Aoife"},
    "Nuala":      {"Nuala"},
    "Róisín":     {"Róisín"},
    "Sorcha":     {"Sorcha"},
    "Treasa":     {"Treasa"},

    # ── Surname particles — lenition/eclipsis of following element ────────────
    # These are handled by the deterministic expander; listed here for
    # irregular stem changes only.
    "Ó":          {"Uí", "Uí"},        # masc gen → Uí, fem gen → Ní (handled separately)
    "Mac":        {"Mhic"},
    "Nic":        {"Nic"},
    "Ní":         {"Ní"},
    "Ua":         {"Uí"},

    # ── Placenames — irregular genitives ─────────────────────────────────────
    "Corcaigh":        {"Chorcaí"},
    "Luimneach":       {"Luimnigh"},
    "Gaillimh":        {"na Gaillimhe"},
    "Doire":           {"Dhoire"},
    "Loch Garman":     {"Loch Garman"},
    "An Clár":         {"An Chláir"},
    "An Mhí":          {"na Mí"},
    "Maigh Eo":        {"Mhaigh Eo"},
    "Tír Eoghain":     {"Thír Eoghain"},
    "Aontroim":        {"Aontroma"},
    "Ard Mhacha":      {"Ard Mhacha"},
    "Fear Manach":     {"Fear Manach"},
    "Dún na nGall":    {"Dhún na nGall"},
    "Sligeach":        {"Shligigh"},
    "Ros Comáin":      {"Ros Comáin"},
    "Liatroim":        {"Liatroma"},
    "Muineachán":      {"Mhuineacháin"},
    "An Cabhán":       {"An Chabháin"},
    "Longfort":        {"Longfoirt"},
    "Uíbh Fhailí":     {"Uíbh Fhailí"},
    "Laois":           {"Laoise"},
    "Cill Dara":       {"Chill Dara"},
    "Cill Mhantáin":   {"Chill Mhantáin"},
    "Loch Laighean":   {"Loch Laighean"},
    "Tiobraid Árann":  {"Thiobraid Árann"},
    "Port Láirge":     {"Phort Láirge"},
    "Ciarraí":         {"Chiarraí"},
    "Ceatharlach":     {"Cheatharlach"},
    "Baile Átha Cliath": {"Bhaile Átha Cliath"},
}

# ── Layer 3 function ──────────────────────────────────────────────────────────

def layer3_manual(entity, label):
    """
    Return set of manual genitive/variant forms for an entity.
    Checks full entity string first, then first token (for multi-token names).
    """
    forms = set()

    # full string match
    if entity in MANUAL_GENITIVE_LEXICON:
        forms.update(MANUAL_GENITIVE_LEXICON[entity])

    # first-token match for multi-token entities (e.g. "Seán Ó Briain" → "Seán")
    tokens = entity.split()
    if len(tokens) > 1 and tokens[0] in MANUAL_GENITIVE_LEXICON:
        for variant in MANUAL_GENITIVE_LEXICON[tokens[0]]:
            # reconstruct full name with mutated first token
            forms.add(variant + " " + " ".join(tokens[1:]))

    # surname particle handling for PER multi-token names
    if label == "PER" and len(tokens) > 1:
        # lenite second token after Mac/Ó if not already lenited
        if tokens[0] in {"Mac", "Mhic"} and len(tokens) > 1:
            second = tokens[1]
            if not second.startswith(("Bh", "Ch", "Dh", "Fh", "Gh", "Mh", "Ph", "Sh", "Th", "bh", "ch", "dh", "fh", "gh", "mh", "ph", "sh", "th")):
                lenited = second[0] + "h" + second[1:] if second[0].islower() else second[0] + "h" + second[1:]
                forms.add("Mhic " + lenited + (" " + " ".join(tokens[2:]) if len(tokens) > 2 else ""))

    return forms

# ── Test ──────────────────────────────────────────────────────────────────────

print("── Layer 3 (Manual genitive lexicon) test ──")
for label, entities in TEST_ENTITIES.items():
    print(f"\n  {label}:")
    for entity in entities:
        forms = layer3_manual(entity, label)
        status = f"{len(forms)} forms: {sorted(forms)}" if forms else "MISS"
        print(f"    {entity:30s} → {status}")

── Layer 3 (Manual genitive lexicon) test ──

  LOC:
    Gaillimh                       → 1 forms: ['na Gaillimhe']
    Corcaigh                       → 1 forms: ['Chorcaí']
    Éire                           → MISS
    Doire                          → 1 forms: ['Dhoire']
    Baile Átha Cliath              → 1 forms: ['Bhaile Átha Cliath']

  PER:
    Seán Ó Briain                  → 1 forms: ['Sheáin Ó Briain']
    Tomás Mac Cárthaigh            → 1 forms: ['Thomáis Mac Cárthaigh']
    Máire Ní Bhriain               → 1 forms: ['Mháire Ní Bhriain']
    Pádraig Ó Néill                → 1 forms: ['Phádraig Ó Néill']

  ORG:
    Sinn Féin                      → MISS
    Fianna Fáil                    → MISS
    RTÉ                            → MISS


In [12]:
# CELL 5 — Layer 4: WikiAnn harvest lookup

# WikiAnn Irish (unimelb-nlp/wikiann, ga config) was pre-harvested and filtered
# in an earlier session. Results:
#   - 187 Irish-marked PER entities (Ó, Ní, Mac, Nic, Ua, Uí, de markers)
#   - 1,485 novel LOC entities (not yet filtered for Irish orthographic markers)
# These are nominative forms only — the deterministic expander (Layer 5) handles
# mutation of these forms.

import subprocess
subprocess.run  # noqa — subprocess imported in earlier cells if needed

from datasets import load_dataset
import re

# ── Irish marker filters ──────────────────────────────────────────────────────

IRISH_PER_MARKERS  = re.compile(
    r'\b(Ó|Uí|Ní|Ua|Mac|Mhic|Nic|de|De|Ó\'|O\')\b'
)
IRISH_LOC_MARKERS  = re.compile(
    r'[áéíóúÁÉÍÓÚ]|'
    r'\b(Abhainn|Baile|Beal|Béal|Bóthar|Carraig|Cill|Cnoc|'
    r'Droichead|Dún|Gleann|Inis|Loch|Maigh|Mullach|Port|'
    r'Ráth|Ros|Sliabh|Sruth|Teach|Tobar|Trá)\b'
)

NOISE_PATTERNS = re.compile(
    r'^\(|'           # starts with parenthesis
    r"^a '\s|"        # Scottish Gaelic a' prefix
    r'^\d|'           # starts with digit
    r'^[A-Z]{1}$'     # single capital letter
)

def is_clean(entity):
    return bool(entity) and not NOISE_PATTERNS.search(entity) and len(entity.split()) >= 1

# ── Load WikiAnn Irish ────────────────────────────────────────────────────────

print("Loading WikiAnn Irish...")
try:
    wikiann = load_dataset("unimelb-nlp/wikiann", "ga", trust_remote_code=True)
except TypeError:
    wikiann = load_dataset("unimelb-nlp/wikiann", "ga")

# extract all entities across splits
raw = {"PER": set(), "LOC": set(), "ORG": set()}

for split in wikiann:
    for example in wikiann[split]:
        tokens = example["tokens"]
        tags   = example["ner_tags"]
        tag_names = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]

        current_tokens, current_label = [], None
        for token, tag_id in zip(tokens, tags):
            tag = tag_names[tag_id]
            if tag.startswith("B-"):
                if current_tokens and current_label:
                    raw[current_label].add(" ".join(current_tokens))
                current_tokens = [token]
                current_label  = tag[2:]
            elif tag.startswith("I-") and current_tokens:
                current_tokens.append(token)
            else:
                if current_tokens and current_label:
                    raw[current_label].add(" ".join(current_tokens))
                current_tokens, current_label = [], None
        if current_tokens and current_label:
            raw[current_label].add(" ".join(current_tokens))

print(f"Raw WikiAnn entities — PER: {len(raw['PER'])}, LOC: {len(raw['LOC'])}, ORG: {len(raw['ORG'])}")

# ── Load existing pools to find novel entities ────────────────────────────────

BASE = "/kaggle/input/datasets/michaelmarkey64"
per_nodes = pd.read_csv(f"{BASE}/irish-ner-kg-consolidated/data/kg/phase_a/per_nodes.csv")
loc_nodes = pd.read_csv(f"{BASE}/irish-ner-kg-consolidated/data/kg/phase_a/loc_nodes.csv")

existing_per = set(per_nodes["label_ga"].dropna().str.strip().str.lower())
existing_loc = set(loc_nodes["label_ga"].dropna().str.strip().str.lower())

# ── Filter ────────────────────────────────────────────────────────────────────

# PER — Irish-marked only, novel only
wikiann_per = {
    e for e in raw["PER"]
    if is_clean(e)
    and IRISH_PER_MARKERS.search(e)
    and e.strip().lower() not in existing_per
}

# LOC — Irish orthographic markers, novel only
wikiann_loc = {
    e for e in raw["LOC"]
    if is_clean(e)
    and IRISH_LOC_MARKERS.search(e)
    and e.strip().lower() not in existing_loc
}

print(f"\nFiltered novel WikiAnn entities:")
print(f"  PER (Irish-marked): {len(wikiann_per)}")
print(f"  LOC (Irish markers): {len(wikiann_loc)}")
print(f"\n  Sample PER: {sorted(wikiann_per)[:10]}")
print(f"  Sample LOC: {sorted(wikiann_loc)[:10]}")

# ── Layer 4 function ──────────────────────────────────────────────────────────

# Build lookup sets for fast membership testing
wikiann_per_lookup = wikiann_per
wikiann_loc_lookup = wikiann_loc

def layer4_wikiann(entity, label):
    """
    Return set containing the WikiAnn nominative form if the entity is in
    the WikiAnn harvest, empty set otherwise.
    Nominative only — Layer 5 (deterministic expander) generates mutations.
    """
    if label == "PER" and entity in wikiann_per_lookup:
        return {entity}
    if label == "LOC" and entity in wikiann_loc_lookup:
        return {entity}
    return set()

# ── Test ──────────────────────────────────────────────────────────────────────

print("\n── Layer 4 (WikiAnn) test ──")
for label, entities in TEST_ENTITIES.items():
    print(f"\n  {label}:")
    for entity in entities:
        forms = layer4_wikiann(entity, label)
        status = f"HIT: {sorted(forms)}" if forms else "MISS (not in WikiAnn harvest)"
        print(f"    {entity:30s} → {status}")

print("\nNote: test entities are from existing pools so WikiAnn misses are expected.")
print(f"WikiAnn adds {len(wikiann_per)} PER + {len(wikiann_loc)} LOC novel nominative forms.")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'unimelb-nlp/wikiann' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading WikiAnn Irish...


README.md: 0.00B [00:00, ?B/s]

ga/validation-00000-of-00001.parquet:   0%|          | 0.00/65.1k [00:00<?, ?B/s]

ga/test-00000-of-00001.parquet:   0%|          | 0.00/65.9k [00:00<?, ?B/s]

ga/train-00000-of-00001.parquet:   0%|          | 0.00/67.6k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Raw WikiAnn entities — PER: 827, LOC: 1532, ORG: 912

Filtered novel WikiAnn entities:
  PER (Irish-marked): 191
  LOC (Irish markers): 694

  Sample PER: ['Ailéin de Diúc', 'Alan Ó Brógáin', 'Ambrós Ó hUiginn', 'An Tiarna Éadbhard Mac Gearailt', 'Anraí Ó Beolláin', 'Anraí Ó Sibhleáin', 'Antóin de Brún', 'Antóin de Nais', 'Aodh Rua Ó Dónaill', 'Aodh Ó Néill']
  Sample LOC: ["A ' Chrálaig", "A ' Chróic", "A ' Mhór Bheinn", 'Abha an tSuláin', 'Abhainn Airgid', 'Abhainn Airgidín', 'Abhainn Buí', 'Abhainn Chaisleán Dhún Dealgan', 'Abhainn Clough', 'Abhainn Glenderamackin']

── Layer 4 (WikiAnn) test ──

  LOC:
    Gaillimh                       → MISS (not in WikiAnn harvest)
    Corcaigh                       → MISS (not in WikiAnn harvest)
    Éire                           → MISS (not in WikiAnn harvest)
    Doire                          → MISS (not in WikiAnn harvest)
    Baile Átha Cliath              → MISS (not in WikiAnn harvest)

  PER:
    Seán Ó Briain                  → MISS (

In [17]:
# KAGGLE CELL 6 — Layer 5: Deterministic morphological expander
# Generates lenition, eclipsis, genitive mutations for any entity as a fallback.

import re

# Lenition map (séimhiú)
LENITION = {
    'b': 'bh', 'c': 'ch', 'd': 'dh', 'f': 'fh', 'g': 'gh',
    'm': 'mh', 'p': 'ph', 's': 'sh', 't': 'th',
    'B': 'Bh', 'C': 'Ch', 'D': 'Dh', 'F': 'Fh', 'G': 'Gh',
    'M': 'Mh', 'P': 'Ph', 'S': 'Sh', 'T': 'Th',
}

# Eclipsis map (urú)
ECLIPSIS = {
    'b': 'mb', 'c': 'gc', 'd': 'nd', 'f': 'bhf', 'g': 'ng',
    'p': 'bp', 't': 'dt',
    'B': 'mB', 'C': 'gC', 'D': 'nD', 'F': 'bhF', 'G': 'nG',
    'P': 'bP', 'T': 'dT',
}

H_PREFIX_TRIGGERS = set('aeiouáéíóúAEIOUÁÉÍÓÚ')


def lenite(token):
    if not token:
        return token
    c = token[0]
    return LENITION[c] + token[1:] if c in LENITION else token


def eclipse(token):
    if not token:
        return token
    c = token[0]
    if c in ECLIPSIS:
        return ECLIPSIS[c] + token[1:]
    if c in H_PREFIX_TRIGGERS:
        return 'n-' + token
    return token


def h_prefix(token):
    if token and token[0] in H_PREFIX_TRIGGERS:
        return 'h' + token
    return token


def apply_mac_mhic(surface):
    """Mac → Mhic wherever it appears; also produce lenited-first + Mhic combined form."""
    if 'Mac ' in surface:
        mhic_form = surface.replace('Mac ', 'Mhic ', 1)
        # also lenite the first token for the fully mutated genitive
        tokens = surface.split()
        lenited_first = lenite(tokens[0])
        if lenited_first != tokens[0]:
            rest = ' '.join(tokens[1:]).replace('Mac ', 'Mhic ', 1)
            return [mhic_form, (lenited_first + ' ' + rest).strip()]
        return [mhic_form]
    if 'mac ' in surface:
        return [surface.replace('mac ', 'mhic ', 1)]
    return []


def apply_nic_nig(surface):
    """Nic → Nig wherever it appears in the surface string."""
    if 'Nic ' in surface:
        return surface.replace('Nic ', 'Nig ', 1)
    return None


def deterministic_expand(surface, entity_type):
    """
    Generate all deterministic morphological variants for an entity surface form.
    Returns a set including the original surface plus all generated variants.

    Rules by entity type:
      PER — lenition and h-prefix on first token only; Mac→Mhic; Nic→Nig
      LOC — lenition, eclipsis, h-prefix on first token; prepositional 'i + eclipsed' form
      ORG — lenition, eclipsis, h-prefix on first token
    """
    forms = {surface}
    tokens = surface.split()
    if not tokens:
        return forms

    first = tokens[0]
    rest = ' '.join(tokens[1:])

    def add(mutated_first):
        if mutated_first and mutated_first != first:
            forms.add((mutated_first + (' ' + rest if rest else '')).strip())

    # lenition — all entity types
    add(lenite(first))

    # h-prefix — all entity types
    add(h_prefix(first))

    # eclipsis — LOC and ORG only, not PER
    if entity_type in ('LOC', 'ORG'):
        eclipsed = eclipse(first)
        add(eclipsed)
        # prepositional form: 'i ' + eclipsed first token (eclipsis already supplies the n)
        if eclipsed != first:
            prep_form = ('i ' + eclipsed + (' ' + rest if rest else '')).strip()
            forms.add(prep_form)

    mac_forms = apply_mac_mhic(surface)
    forms.update(mac_forms)

    nic_form = apply_nic_nig(surface)
    if nic_form:
        forms.add(nic_form)

    return forms


# ── test ──
print("── Layer 5 (Deterministic expander) test ──\n")

TEST_ENTITIES = {
    "LOC": ["Gaillimh", "Corcaigh", "Éire", "Doire", "Baile Átha Cliath"],
    "PER": ["Seán Ó Briain", "Tomás Mac Cárthaigh", "Máire Ní Bhriain", "Pádraig Ó Néill"],
    "ORG": ["Sinn Féin", "Fianna Fáil", "RTÉ"],
}

for etype, entities in TEST_ENTITIES.items():
    print(f"  {etype}:")
    for surface in entities:
        forms = deterministic_expand(surface, etype)
        novel = sorted(forms - {surface})
        if novel:
            print(f"    {surface:30s} → {len(forms)} forms: {novel}")
        else:
            print(f"    {surface:30s} → no new forms generated")
    print()

── Layer 5 (Deterministic expander) test ──

  LOC:
    Gaillimh                       → 4 forms: ['Ghaillimh', 'i nGaillimh', 'nGaillimh']
    Corcaigh                       → 4 forms: ['Chorcaigh', 'gCorcaigh', 'i gCorcaigh']
    Éire                           → 4 forms: ['hÉire', 'i n-Éire', 'n-Éire']
    Doire                          → 4 forms: ['Dhoire', 'i nDoire', 'nDoire']
    Baile Átha Cliath              → 4 forms: ['Bhaile Átha Cliath', 'i mBaile Átha Cliath', 'mBaile Átha Cliath']

  PER:
    Seán Ó Briain                  → 2 forms: ['Sheán Ó Briain']
    Tomás Mac Cárthaigh            → 4 forms: ['Thomás Mac Cárthaigh', 'Thomás Mhic Cárthaigh', 'Tomás Mhic Cárthaigh']
    Máire Ní Bhriain               → 2 forms: ['Mháire Ní Bhriain']
    Pádraig Ó Néill                → 2 forms: ['Phádraig Ó Néill']

  ORG:
    Sinn Féin                      → 2 forms: ['Shinn Féin']
    Fianna Fáil                    → 4 forms: ['Fhianna Fáil', 'bhFianna Fáil', 'i bhFianna Fáil']
    

In [19]:
import subprocess
subprocess.run(["apt-get", "install", "-y", "-q", "aspell", "aspell-ga"], check=True)

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  dictionaries-common libaspell15 libtext-iconv-perl
Suggested packages:
  aspell-doc spellutils wordlist
The following NEW packages will be installed:
  aspell aspell-ga dictionaries-common libaspell15 libtext-iconv-perl
0 upgraded, 5 newly installed, 0 to remove and 90 not upgraded.
Need to get 935 kB of archives.
After this operation, 3,752 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libtext-iconv-perl amd64 1.7-7build3 [14.3 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libaspell15 amd64 0.60.8-4build1 [325 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 dictionaries-common all 1.28.14 [185 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 aspell amd64 0.60.8-4build1 [87.7 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 aspell-ga all 0.50-4-6 [3

CompletedProcess(args=['apt-get', 'install', '-y', '-q', 'aspell', 'aspell-ga'], returncode=0)

In [21]:
# KAGGLE CELL 7 — Layer 6: aspell-ga validity filter
# Filters generated forms against the Irish spell-checker word list.
# Removes forms that are not valid Irish words, reducing noise from
# the deterministic expander. Multi-token forms are checked token by token.

import subprocess

def setup_aspell():
    """Install aspell-ga if not already present."""
    result = subprocess.run(
        ["aspell", "-l", "ga", "list"],
        input="", capture_output=True, text=True
    )
    if result.returncode != 0:
        subprocess.run(
            ["apt-get", "install", "-y", "-q", "aspell", "aspell-ga"],
            capture_output=True
        )

setup_aspell()


def aspell_check_token(token):
    """Return True if token is accepted by aspell-ga."""
    token = token.strip().rstrip('.,;:')
    if not token:
        return True
    result = subprocess.run(
        ["aspell", "-l", "ga", "list"],
        input=token + "\n",
        capture_output=True, text=True
    )
    # aspell list outputs misspelled words; empty output = word is valid
    return result.stdout.strip() == ""


def aspell_check_surface(surface):
    """
    Check a surface form (possibly multi-token) against aspell-ga.
    Returns True if all content tokens are valid Irish.
    Skips tokens that are: all-caps acronyms, numbers, punctuation-only,
    or Irish mutation prefixes (i, a, na, an, etc.).
    """
    SKIP_TOKENS = {'i', 'a', 'na', 'an', 'in', 'ag', 'ar', 'as', 'de', 'do',
                   'faoi', 'le', 'ó', 'thar', 'um', 'sa', 'sna'}

    tokens = surface.split()
    for token in tokens:
        clean = token.strip("'-")
        if not clean:
            continue
        # skip acronyms
        if clean.upper() == clean and len(clean) <= 5:
            continue
        # skip prepositions and particles
        if clean.lower() in SKIP_TOKENS:
            continue
        # skip non-alpha tokens
        if not any(c.isalpha() for c in clean):
            continue
        # skip proper noun tokens — aspell won't have them regardless
        if clean[0].isupper():
            continue
        # skip n- and h- prefixed vowel forms (valid mutation prefixes)
        if re.match(r'^[nh]-[aeiouáéíóúAEIOUÁÉÍÓÚ]', clean):
            continue
        if not aspell_check_token(clean):
            return False
    return True


def filter_forms_aspell(forms, original_surface):
    """
    Given a set of generated forms, return those that pass aspell-ga.
    Always retains the original surface regardless of aspell result.
    """
    retained = {original_surface}
    for form in forms:
        if form == original_surface:
            continue
        if aspell_check_surface(form):
            retained.add(form)
    return retained


# ── test ──
print("── Layer 6 (aspell-ga filter) test ──\n")

# use the same test set with pre-generated forms from Layer 5
from collections import defaultdict

TEST_FORMS = {
    "LOC": {
        "Gaillimh":         {"Gaillimh", "Ghaillimh", "nGaillimh", "i nGaillimh"},
        "Corcaigh":         {"Corcaigh", "Chorcaigh", "gCorcaigh", "i gCorcaigh"},
        "Éire":             {"Éire", "hÉire", "n-Éire", "i n-Éire"},
        "Doire":            {"Doire", "Dhoire", "nDoire", "i nDoire"},
        "Baile Átha Cliath": {"Baile Átha Cliath", "Bhaile Átha Cliath",
                              "mBaile Átha Cliath", "i mBaile Átha Cliath"},
    },
    "PER": {
        "Seán Ó Briain":       {"Seán Ó Briain", "Sheán Ó Briain"},
        "Tomás Mac Cárthaigh": {"Tomás Mac Cárthaigh", "Thomás Mac Cárthaigh",
                                "Tomás Mhic Cárthaigh", "Thomás Mhic Cárthaigh"},
        "Máire Ní Bhriain":    {"Máire Ní Bhriain", "Mháire Ní Bhriain"},
        "Pádraig Ó Néill":     {"Pádraig Ó Néill", "Phádraig Ó Néill"},
    },
    "ORG": {
        "Sinn Féin":   {"Sinn Féin", "Shinn Féin"},
        "Fianna Fáil": {"Fianna Fáil", "Fhianna Fáil", "bhFianna Fáil", "i bhFianna Fáil"},
        "RTÉ":         {"RTÉ"},
    },
}

for etype, entities in TEST_FORMS.items():
    print(f"  {etype}:")
    for surface, forms in entities.items():
        retained = filter_forms_aspell(forms, surface)
        rejected = forms - retained
        print(f"    {surface:30s} → kept {len(retained)}/{len(forms)}", end="")
        if rejected:
            print(f"  | rejected: {sorted(rejected)}", end="")
        print()
    print()

── Layer 6 (aspell-ga filter) test ──

  LOC:
    Gaillimh                       → kept 4/4
    Corcaigh                       → kept 4/4
    Éire                           → kept 4/4
    Doire                          → kept 4/4
    Baile Átha Cliath              → kept 4/4

  PER:
    Seán Ó Briain                  → kept 2/2
    Tomás Mac Cárthaigh            → kept 4/4
    Máire Ní Bhriain               → kept 2/2
    Pádraig Ó Néill                → kept 2/2

  ORG:
    Sinn Féin                      → kept 2/2
    Fianna Fáil                    → kept 4/4
    RTÉ                            → kept 1/1



In [26]:
# KAGGLE CELL 8 (updated) — Layer 7: expand_entity() combinator + full coverage report
# Chains all layers in priority order and reports coverage across entity pools.
# Fix: UD lookup now checks individual tokens for PER (treebank lemmas are single tokens).

from collections import defaultdict
import pandas as pd


 
# =============================================================================
# EXPAND_ENTITY (updated) — place this in Cell 8
# =============================================================================
 
def expand_entity(surface, entity_type):
    """
    Run all morphological layers for a single entity surface form.
    Returns a set of all attested and generated variant forms.
 
    Layer priority:
      1  Logainm         — LOC attested forms (highest quality)
      2  UD Irish-IDT    — attested variants from treebank
      3  Manual genitive — irregular genitive lexicon
      4  WikiAnn         — novel nominative forms (used upstream in pool, not here)
      5  Deterministic   — rule-based mutation fallback
      6  aspell-ga       — validity filter on deterministic output
    """
    forms = {surface}
    layers_hit = []
 
    # Layer 1: Logainm (LOC only)
    if entity_type == 'LOC':
        lg_forms = logainm_lookup.get(surface, set())
        if lg_forms:
            forms.update(lg_forms)
            layers_hit.append('logainm')
 
    # Layer 2: UD Irish-IDT
    # For multi-token entities check each token individually and reconstruct
    # full-name variants with the mutated token in position.
    ud_hit = False
    tokens = surface.split()
    if len(tokens) == 1:
        ud_forms = ud_lookup.get(surface, set())
        if ud_forms:
            forms.update(ud_forms)
            ud_hit = True
    else:
        for i, token in enumerate(tokens):
            token_variants = ud_lookup.get(token, set())
            novel_variants = token_variants - {token}
            for variant in novel_variants:
                # skip all-caps variants
                if variant == variant.upper() and len(variant) > 2:
                    continue
                reconstructed = tokens[:i] + [variant] + tokens[i+1:]
                # skip if any non-first token lost its capitalisation
                valid = True
                for j, (orig, new) in enumerate(zip(tokens, reconstructed)):
                    if j == i:
                        continue
                    if orig[0].isupper() and new[0].islower():
                        valid = False
                        break
                if valid:
                    forms.add(' '.join(reconstructed))
                    ud_hit = True
    if ud_hit:
        layers_hit.append('ud')
 
    # Layer 3: Manual genitive lexicon
    manual_forms = layer3_manual(surface, entity_type)
    new_manual = manual_forms - forms
    if new_manual:
        forms.update(new_manual)
        layers_hit.append('manual')
 
    # Layer 5: Deterministic expander
    det_forms = deterministic_expand(surface, entity_type)
    new_det = det_forms - forms
 
    # Layer 6: aspell-ga filter on deterministic output only
    if new_det:
        filtered_det = filter_forms_aspell(new_det | {surface}, surface) - {surface}
        if filtered_det:
            forms.update(filtered_det)
            layers_hit.append('deterministic')
 
    return forms, layers_hit


def coverage_report(nodes_df, entity_type, label_col='label_ga'):
    """
    Run expand_entity over all entities in a node dataframe.
    Deduplicates on label_col before running.
    Reports: expansion factor, layer hit rates, slot coverage.
    """
    # deduplicate on label_ga
    nodes_df = nodes_df.dropna(subset=[label_col]).drop_duplicates(subset=[label_col])

    results = []
    for _, row in nodes_df.iterrows():
        surface = str(row.get(label_col, '') or '').strip()
        if not surface or surface.lower() == 'nan':
            continue
        forms, layers = expand_entity(surface, entity_type)
        results.append({
            'surface': surface,
            'n_forms': len(forms),
            'forms': forms,
            'layers': layers,
            'layer_count': len(layers),
        })

    df = pd.DataFrame(results)
    if df.empty:
        print(f"  No entities found for {entity_type}")
        return df

    total = len(df)
    print(f"\n{'─'*55}")
    print(f"  {entity_type} — {total} entities (deduplicated)")
    print(f"{'─'*55}")
    print(f"  Expansion factor  : {df['n_forms'].mean():.2f} forms/entity (mean)")
    print(f"  Max forms         : {df['n_forms'].max()}")
    print(f"  Min forms         : {df['n_forms'].min()}")
    print(f"  Single-form (1)   : {(df['n_forms'] == 1).sum()} ({(df['n_forms'] == 1).mean()*100:.1f}%)")
    print(f"  2+ forms          : {(df['n_forms'] >= 2).sum()} ({(df['n_forms'] >= 2).mean()*100:.1f}%)")
    print(f"  5+ forms          : {(df['n_forms'] >= 5).sum()} ({(df['n_forms'] >= 5).mean()*100:.1f}%)")

    print(f"\n  Layer hit rates:")
    for layer in ['logainm', 'ud', 'manual', 'deterministic']:
        if entity_type != 'LOC' and layer == 'logainm':
            continue
        n_hit = df['layers'].apply(lambda x: layer in x).sum()
        print(f"    {layer:15s}: {n_hit}/{total} ({n_hit/total*100:.1f}%)")

    print(f"\n  Sample high-coverage entities (5+ forms):")
    high = df[df['n_forms'] >= 5].head(5)
    for _, r in high.iterrows():
        print(f"    {r['surface']:25s} → {sorted(r['forms'])}")

    print(f"\n  Sample single-form entities (coverage gap):")
    low = df[df['n_forms'] == 1].head(5)
    for _, r in low.iterrows():
        print(f"    {r['surface']}")

    return df


# ── run coverage report across all entity pools ──
print("── Layer 7: Full pipeline coverage report ──")

BASE = "/kaggle/input/datasets/michaelmarkey64"

per_nodes = pd.read_csv(f"{BASE}/irish-ner-kg-consolidated/data/kg/phase_a/per_nodes.csv")
loc_nodes = pd.read_csv(f"{BASE}/irish-ner-kg-consolidated/data/kg/phase_a/loc_nodes.csv")
org_nodes = pd.read_csv(f"{BASE}/irish-ner-kg-consolidated/data/kg/phase_a/org_nodes.csv")

per_results = coverage_report(per_nodes, 'PER', label_col='label_ga')
loc_results = coverage_report(loc_nodes, 'LOC', label_col='label_ga')
org_results = coverage_report(org_nodes, 'ORG', label_col='label_ga')

# ── combined summary ──
print(f"\n{'='*55}")
print("  COMBINED SUMMARY")
print(f"{'='*55}")
all_results = pd.concat([
    per_results.assign(type='PER'),
    loc_results.assign(type='LOC'),
    org_results.assign(type='ORG'),
], ignore_index=True)

print(f"  Total entities     : {len(all_results)}")
print(f"  Mean forms/entity  : {all_results['n_forms'].mean():.2f}")
print(f"  Entities with 1 form (no expansion): "
      f"{(all_results['n_forms'] == 1).sum()} "
      f"({(all_results['n_forms'] == 1).mean()*100:.1f}%)")
print(f"  Entities with 2+ forms: "
      f"{(all_results['n_forms'] >= 2).sum()} "
      f"({(all_results['n_forms'] >= 2).mean()*100:.1f}%)")

── Layer 7: Full pipeline coverage report ──

───────────────────────────────────────────────────────
  PER — 811 entities (deduplicated)
───────────────────────────────────────────────────────
  Expansion factor  : 1.83 forms/entity (mean)
  Max forms         : 6
  Min forms         : 1
  Single-form (1)   : 302 (37.2%)
  2+ forms          : 509 (62.8%)
  5+ forms          : 11 (1.4%)

  Layer hit rates:
    ud             : 131/811 (16.2%)
    manual         : 23/811 (2.8%)
    deterministic  : 423/811 (52.2%)

  Sample high-coverage entities (5+ forms):
    Proinsias Mac Aodhagáin   → ['Phroinsias Mac Aodhagáin', 'Phroinsias Mhic Aodhagáin', 'Proinsias Mac Aodhagáin', 'Proinsias Mhic Aodhagáin', 'Proinsiis Mac Aodhagáin']
    Seán Mac Brádaigh         → ['Seán Mac Bhrádaigh', 'Seán Mac Brádaigh', 'Seán Mhic Brádaigh', 'Sheáin Mac Brádaigh', 'Sheán Mac Brádaigh', 'Sheán Mhic Brádaigh']
    Breandán Mac Fheorais     → ['Bhreandáin Mac Fheorais', 'Bhreandán Mac Fheorais', 'Bhreandán Mh

In [27]:
# KAGGLE CELL 9 — Serialise pipeline assets
# Run once in the current session after all lookup tables are built.
# Upload the output folder to a new Kaggle dataset: morph-pipeline-assets

import pickle, json, os

ASSETS_DIR = "/kaggle/working/pipeline_assets"
os.makedirs(ASSETS_DIR, exist_ok=True)

# 1. Logainm lookup — dict[str, set[str]] → pickle
with open(f"{ASSETS_DIR}/logainm_lookup.pkl", "wb") as f:
    pickle.dump(dict(logainm_lookup), f)
print(f"logainm_lookup.pkl — {len(logainm_lookup)} entries")

# 2. UD lookup — dict[str, set[str]] → pickle
with open(f"{ASSETS_DIR}/ud_lookup.pkl", "wb") as f:
    pickle.dump(dict(ud_lookup), f)
print(f"ud_lookup.pkl      — {len(ud_lookup)} entries")

# 3. Manual genitive lexicon — JSON (human-readable, easy to extend)
# Convert sets to sorted lists for JSON serialisation
manual_json = {k: sorted(v) for k, v in MANUAL_GENITIVE_LEXICON.items()}
with open(f"{ASSETS_DIR}/manual_genitive_lexicon.json", "w", encoding="utf-8") as f:
    json.dump(manual_json, f, ensure_ascii=False, indent=2)
print(f"manual_genitive_lexicon.json — {len(MANUAL_GENITIVE_LEXICON)} entries")

# 4. WikiAnn entities — plain text, one per line
wikiann_per_path = f"{ASSETS_DIR}/wikiann_per.txt"
wikiann_loc_path = f"{ASSETS_DIR}/wikiann_loc.txt"

# wikiann_per and wikiann_loc should be sets/lists from your Layer 4 cell
# adjust variable names if yours differ
with open(wikiann_per_path, "w", encoding="utf-8") as f:
    for entity in sorted(wikiann_per):
        f.write(entity + "\n")
print(f"wikiann_per.txt    — {len(wikiann_per)} entities")

with open(wikiann_loc_path, "w", encoding="utf-8") as f:
    for entity in sorted(wikiann_loc):
        f.write(entity + "\n")
print(f"wikiann_loc.txt    — {len(wikiann_loc)} entities")

# 5. Verify all files written
print(f"\nAssets written to {ASSETS_DIR}:")
for fname in sorted(os.listdir(ASSETS_DIR)):
    size = os.path.getsize(f"{ASSETS_DIR}/{fname}")
    print(f"  {fname:40s} {size:>8,} bytes")

print("\nNext step: download pipeline_assets/ and upload to Kaggle dataset 'morph-pipeline-assets'")

logainm_lookup.pkl — 117 entries
ud_lookup.pkl      — 1792 entries
manual_genitive_lexicon.json — 69 entries
wikiann_per.txt    — 191 entities
wikiann_loc.txt    — 694 entities

Assets written to /kaggle/working/pipeline_assets:
  logainm_lookup.pkl                          4,509 bytes
  manual_genitive_lexicon.json                2,489 bytes
  ud_lookup.pkl                              41,167 bytes
  wikiann_loc.txt                            11,397 bytes
  wikiann_per.txt                             4,204 bytes

Next step: download pipeline_assets/ and upload to Kaggle dataset 'morph-pipeline-assets'
